# **AI Agent**
---

In this doc file, we will talk about the agent implementation into the **src/lunarAI.py** file.
It is cut into three `class` that does a part of the agent job.

We will begin with the `ANN()` class. It's goal is to create the blank sheet of the neural networking algorithm. In **Reinforcement Learning**, when you chose to build a **Deep-Q learning** system, you must create a kind of blank sheet that will be filled with agent decision, and it will then read it to decide at the next time what could be the best choice in the same situation. The goal of `ANN()` is to manage the **learning sheet**. It looks like this:

---


### **Artificial Neural Network**
**Artificial Neural Network** for **Q-value** approximation
This network takes a state as input and outputs **Q-values** for each action.
It consists of 3 fully connected layers with ReLU activations.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class ANN(nn.Module):

    def __init__(self, state_size, action_size, seed=42):
        super(ANN, self).__init__()
        self.seed = torch.manual_seed(seed)
        self.fc1 = nn.Linear(state_size, 64)
        self.fc2 = nn.Linear(64, 64)
        self.fc3 = nn.Linear(64, action_size)

    def forward(self, state):
        x = self.fc1(state)
        x = F.relu(x)
        x = self.fc2(x)
        x = F.relu(x)
        return self.fc3(x)

`forward()` pass through the network, it takes an **input state tensor** and returns **Q-values** for each other action.

In [ ]:
def forward(self, state):
    x = self.fc1(state)
    x = F.relu(x)
    x = self.fc2(x)
    x = F.relu(x)
    return self.fc3(x)

---

### **Replay Memory**
Then, to learn from the **experiences sheet** and put our decision the right cell, we created a `ReplayMemory()` class. It initializes an empty `memory` attribute with a given `capacity` attribute.

This buffer stores transitions `(state, action, reward, next_state, done)` to break
correlation between training samples and improve stability.

In [ ]:
import random
import numpy as np

class ReplayMemory(object):

    def __init__(self, capacity):
        self.capacity = capacity
        self.memory = []

    def push(self, event):
        self.memory.append(event)
        if len(self.memory) > self.capacity:
            del self.memory[0]

    def sample(self, batch_size):
        experiences = random.sample(self.memory, batch_size)
        states = torch.from_numpy(np.vstack([e[0] for e in experiences if e is not None])).float()
        actions = torch.from_numpy(np.vstack([e[1] for e in experiences if e is not None])).long()
        rewards = torch.from_numpy(np.vstack([e[2] for e in experiences if e is not None])).float()
        next_states = torch.from_numpy(np.vstack([e[3] for e in experiences if e is not None])).float()
        dones = torch.from_numpy(np.vstack([e[4] for e in experiences if e is not None]).astype(np.uint8)).float()
        return states, actions, rewards, next_states, dones

`push()` add a new experience to the memory. It takes a tuple of `(state, action, reward, next_state, done)`. When memory exceeds capacity, the oldest experience is removed.

In [ ]:
def push(self, event):
    self.memory.append(event)
    if len(self.memory) > self.capacity:
        del self.memory[0]

`sample()` a random batch of experiences from memory. It takes a number of experiences to sample. It returns a tuple of `(states, actions, rewards, next_states, dones)` as **torch tensors**

In [ ]:
def sample(self, batch_size):
    experiences = random.sample(self.memory, batch_size)
    states = torch.from_numpy(np.vstack([e[0] for e in experiences if e is not None])).float()
    actions = torch.from_numpy(np.vstack([e[1] for e in experiences if e is not None])).long()
    rewards = torch.from_numpy(np.vstack([e[2] for e in experiences if e is not None])).float()
    next_states = torch.from_numpy(np.vstack([e[3] for e in experiences if e is not None])).float()
    dones = torch.from_numpy(np.vstack([e[4] for e in experiences if e is not None]).astype(np.uint8)).float()
    return states, actions, rewards, next_states, dones

---

### **Agent Class**
Finally, we have the `Agent()` class. This is the main class that orchestrates the learning process. It manages both the **local Q-network** (which is actively trained) and the **target Q-network** (which is used to compute stable Q-targets). The agent has five key methods:

1. **`__init__()`**: Initializes both networks, the optimizer, and the replay memory
2. **`step()`**: Stores experiences and triggers learning every 4 steps
3. **`get_action()`**: Selects actions using epsilon-greedy exploration
4. **`learn()`**: Computes the Q-learning loss and updates the local network
5. **`soft_update()`**: Gradually updates the target network parameters


This **agent** uses two **neural networks** (local and target) to learn optimal actions
in the **LunarLander** environment using the **Deep Q-Network** (DQN) algorithm.

`step()` stores experience and trigger learning every 4 steps. It takes the current state, the taken action, a received reward, a next_state and a data that tell whether episode is finished.

In [ ]:
def step(self, state, action, reward, next_state, done):
     self.memory.push((state, action, reward, next_state, done))
     self.t_step = (self.t_step + 1) % 4
     if self.t_step == 0 and len(self.memory.memory) > 150:
         experiences = self.memory.sample(150)
         self.learn(experiences, gamma=0.99)

`get_action()` selects an action using epsilon-greedy exploration. It takes a current state and exploration probability. It returns selected action (0, 1, 2, or 3 for LunarLander).

In [ ]:
def get_action(self, state, epsilon):
    state = torch.from_numpy(state).float().unsqueeze(0)
    self.local_qnetwork.eval()
    with torch.no_grad():
        action_values = self.local_qnetwork(state)
    self.local_qnetwork.train()
    if random.random() > epsilon:
        return np.argmax(action_values.cpu().data.numpy())
    else:
        return random.choice(np.arange(self.action_size))

`learn()` updates the **Q-network** using a batch of experiences. This method computes the **Q-learning** loss using the **Bellman equation**. It takes experiences and a discount factor.

In [1]:
def learn(self, experiences, gamma):
    states, actions, rewards, next_states, dones = experiences
    next_q_targets = self.target_qnetwork(next_states).detach().max(1)[0].unsqueeze(1)
    q_targets = rewards + (gamma * next_q_targets * (1 - dones))
    q_expected = self.local_qnetwork(states).gather(1, actions)
    loss = F.mse_loss(q_expected, q_targets)
    self.optimizer.zero_grad()
    loss.backward()
    self.optimizer.step()
    self.soft_update(self.local_qnetwork, self.target_qnetwork,
                     self.interpolation_parameter)

`soft_update()` of target network parameters. Instead of copying weights directly, we perform a weighted average:
```sh
θ_target = τ * θ_local + (1 - τ) * θ_target
```
This provides more stable learning compared to hard updates. It takes local **Q-network** being trained, target **Q-network** for computingstable targets and soft update parameter (τ, typically 1e-3).

In [2]:
def soft_update(self, local_qnetwork, target_qnetwork, interpolation_parameter):
    for target_params, local_params in zip(target_qnetwork.parameters(),
                                           local_qnetwork.parameters()):
        target_params.data.copy_(interpolation_parameter * local_params.data +
                                (1.0 - interpolation_parameter) * target_params.data)

Here's the complete `Agent()` class:

In [ ]:
import torch.optim as optim

class Agent():

    def __init__(self, state_size, action_size, learning_rate=5e-4,
                 replay_buffer_size=100000, interpolation_parameter=1e-3):
        self.state_size = state_size
        self.action_size = action_size
        self.interpolation_parameter = interpolation_parameter
        self.local_qnetwork = ANN(state_size, action_size)
        self.target_qnetwork = ANN(state_size, action_size)
        self.target_qnetwork.load_state_dict(self.local_qnetwork.state_dict())
        self.optimizer = optim.Adam(self.local_qnetwork.parameters(), lr=learning_rate)
        self.memory = ReplayMemory(replay_buffer_size)
        self.t_step = 0

    def step(self, state, action, reward, next_state, done):
        self.memory.push((state, action, reward, next_state, done))
        self.t_step = (self.t_step + 1) % 4
        if self.t_step == 0 and len(self.memory.memory) > 150:
            experiences = self.memory.sample(150)
            self.learn(experiences, gamma=0.99)

    def get_action(self, state, epsilon):
        state = torch.from_numpy(state).float().unsqueeze(0)
        self.local_qnetwork.eval()
        with torch.no_grad():
            action_values = self.local_qnetwork(state)
        self.local_qnetwork.train()
        if random.random() > epsilon:
            return np.argmax(action_values.cpu().data.numpy())
        else:
            return random.choice(np.arange(self.action_size))

    def learn(self, experiences, gamma):
        states, actions, rewards, next_states, dones = experiences
        next_q_targets = self.target_qnetwork(next_states).detach().max(1)[0].unsqueeze(1)
        q_targets = rewards + (gamma * next_q_targets * (1 - dones))
        q_expected = self.local_qnetwork(states).gather(1, actions)
        loss = F.mse_loss(q_expected, q_targets)
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()
        self.soft_update(self.local_qnetwork, self.target_qnetwork,
                         self.interpolation_parameter)

    def soft_update(self, local_qnetwork, target_qnetwork, interpolation_parameter):
        for target_params, local_params in zip(target_qnetwork.parameters(),
                                               local_qnetwork.parameters()):
            target_params.data.copy_(interpolation_parameter * local_params.data +
                                    (1.0 - interpolation_parameter) * target_params.data)

---

## Summary

The three main classes work together to implement the **Deep Q-Learning algorithm**:

- **`ANN()`**: The neural network that approximates Q-values
- **`ReplayMemory()`**: Stores and samples experiences to break temporal correlations
- **`Agent()`**: Orchestrates the learning process by managing both networks and the training loop

The key innovation is using two networks: the **local network** is trained on the loss, while the **target network** provides stable Q-targets. The soft update mechanism ensures smooth parameter transitions and better convergence.